# Helpers for Dataset Generation
sample_land_coordinates(num_points)

## Create BW country/ocean map for coordinate validation
to easily check if a coordinate is land or ocean

In [ ]:
# imports
import os
import pandas as pd
import geopandas as gpd

# import my local module (pain2map) - this is a bit hacky but it works for now
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from pain2map import Util

In [ ]:
# constants / config
BASE_PATH = os.path.join("..", "data")
WORLD_PATH = os.path.join(BASE_PATH, "countries_map.zip")
# dummy data paths
DUMMY_BASE_PATH = os.path.join(BASE_PATH, "dummy")
BWLO_PATH = os.path.join(DUMMY_BASE_PATH, "bwlo_map.png")

os.makedirs(DUMMY_BASE_PATH, exist_ok=True)

In [ ]:
# Helper Functions

def create_bw_land_ocean_dataset() -> pd.DataFrame:
    """Creates a dummy dataset for land and ocean visualization."""
    countries = Util.get_sova3_countries(WORLD_PATH)
    data = []
    for country, sova3 in countries.items():
        data.append({
            'sov_a3': sova3,
            'Country': country,
            'value': 1
        })
    return pd.DataFrame(data)

def create_bw_land_ocean_texture(output_path: str, width: int = 1000, height: int = 1500, dpi: int = 100):
    """Creates a black and white land/ocean texture map with the given parameters."""
    bwlo_data = create_bw_land_ocean_dataset()
    Util.generate_map(
        data=bwlo_data,
        world_path=WORLD_PATH,
        output_path=output_path,
        args_value_col="value",
        args_code_col="sov_a3",
        width=width,
        dpi=dpi,
        cmap="grey",
        projection="PlateCarree"
    )


In [ ]:
# Create the bwlo texture and save it to a png file (activate if the file does not exist yet)
if True:
    create_bw_land_ocean_texture(BWLO_PATH)

## Create a method to sample random land coordinates according to the map texture generated above
sample_land_coordinates(num_points)

In [ ]:
# imports
from typing import List, Tuple
from PIL import Image

import random

In [ ]:
# constants / config
LAND_COORDINATES_PATH = os.path.join(DUMMY_BASE_PATH, "land_coordinates.csv")

In [ ]:
# methods to get coordinate data from a BWLO texture
def get_coordinates(bwlo_path: str, num_points: int = 100) -> List[Tuple[float, float, bool]]:
    """Returns a list of tuples containing (latitude, longitude, is_land) for random points on the BWLO texture."""

    bwlo_texture = Image.open(bwlo_path)
    bwlo_width, bwlo_height = bwlo_texture.size

    data = []
    while len(data) < num_points:
        # get random point on texture
        x = random.uniform(0, bwlo_width)
        y = random.uniform(0, bwlo_height)

        # check if it's land (black) or ocean (white)
        r, g, b, _ = bwlo_texture.getpixel((x, y))
        #if :  # black pixel (land)
        data.append((y, x, (r + g + b) < 128))  # if the pixel is dark, consider it land (True), otherwise ocean (False)
    return data

def create_land_coordinate_dataset(bwlo_path: str) -> pd.DataFrame:
    """Create a dataset of land coordinates from the BWLO texture at the provided path."""
    bwlo_texture = Image.open(bwlo_path)
    land_pixels = []
    width, height = bwlo_texture.size
    for x in range(0, width):
        for y in range(0, height):
            r, g, b, _ = bwlo_texture.getpixel((x, y))
            if (r + g + b) < 128:  # black pixel (land)
                land_pixels.append({
                    "x": x,
                    "y": y,
                    "pixel": (r, g, b)
                })
    return pd.DataFrame(land_pixels)

In [ ]:
# Create the land coordinate dataset and save it to a CSV file (activate if the file does not exist yet)
if True:
    df_land_coordinates = create_land_coordinate_dataset(BWLO_PATH)
    df_land_coordinates.to_csv(LAND_COORDINATES_PATH, index=False)

In [ ]:
# expose the sampling function for use in code cells below
def sample_land_coordinates(num_points: int = 100) -> List[Tuple[float, float]]:
    """Returns a list of tuples containing (x, y) for random coordinates according to the BWLO land_coordinates dataset."""
    df_land_coordinates = pd.read_csv(LAND_COORDINATES_PATH)
    df_samples = df_land_coordinates.sample(n=num_points)
    return list(zip(df_samples["x"], df_samples["y"]))

# Create Dummy Datasets

In [ ]:
# general imports
from typing import Optional, Dict, Union

import os
import enum
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# general constants / config
DUMMY_DATASET_BASE_PATH = os.path.join("..", "data", "dummy", "data_types")
DEFAULT_NUM_POINTS = 1_000

## Helpers

In [ ]:
class DummyDataOrigin(enum.Enum):
    Env_Natural = "EnvNat"
    Env_Antropogenic = "EnvAntro"
    EmotionalHealth = "Emo"
    PhysicalHealth = "Phys"
    Socioeconomic = "Socioeco"

    def __str__(self):
        return self.value

In [ ]:
class DummyDataType(enum.Enum):
    # Environmental - Natural
    Fire = ("fire", DummyDataOrigin.Env_Natural)
    Floods = ("floods", DummyDataOrigin.Env_Natural)
    Earthquakes = ("earthquakes", DummyDataOrigin.Env_Natural)
    VolcanicEruptions = ("volcanic-eruptions", DummyDataOrigin.Env_Natural)
    Tornados = ("tornados", DummyDataOrigin.Env_Natural)

    # Environmental - Antropogenic
    Deforestation = ("deforestation", DummyDataOrigin.Env_Antropogenic)
    SpeciesExtinction = ("species-extinction", DummyDataOrigin.Env_Antropogenic)
    Smog = ("smog", DummyDataOrigin.Env_Antropogenic)
    PlasticPollution = ("plastic-pollution", DummyDataOrigin.Env_Antropogenic)
    HeavyMetalContamination = ("heavy-metal-contamination", DummyDataOrigin.Env_Antropogenic)
    RareEarthMining = ("rare-earth-mining", DummyDataOrigin.Env_Antropogenic)

    # Emotional Health
    Depression = ("depression", DummyDataOrigin.EmotionalHealth)
    Anxiety = ("anxiety", DummyDataOrigin.EmotionalHealth)
    Suicide = ("suicide", DummyDataOrigin.EmotionalHealth)
    Isolation = ("isolation", DummyDataOrigin.EmotionalHealth)
    CommunityResilience = ("community-resilience", DummyDataOrigin.EmotionalHealth)

    # Physical Health
    ChronicDisease = ("chronic-disease", DummyDataOrigin.PhysicalHealth)
    ChronicPain = ("chronic-pain", DummyDataOrigin.PhysicalHealth)
    CancerRate = ("cancer-rate", DummyDataOrigin.PhysicalHealth)
    AutoimmuneDisorders = ("autoimmune-disorders", DummyDataOrigin.PhysicalHealth)
    RespiratoryIllness = ("respiratory-illness", DummyDataOrigin.PhysicalHealth)
    NeurologicalConditions = ("neurological-conditions", DummyDataOrigin.PhysicalHealth)
    Obesity = ("obesity", DummyDataOrigin.PhysicalHealth)

    # Socioeconomic
    GDP = ("gdp", DummyDataOrigin.Socioeconomic)
    Gini = ("gini", DummyDataOrigin.Socioeconomic)
    PovertyIndex = ("poverty-index", DummyDataOrigin.Socioeconomic)
    FoodInsecurityIndex = ("food-insecurity-index", DummyDataOrigin.Socioeconomic)
    HealthcareAccess = ("healthcare-access", DummyDataOrigin.Socioeconomic)
    ExtractiveIndustryEmployment = ("extractive-industry-employment", DummyDataOrigin.Socioeconomic)
    MigrationRate = ("migration-rate", DummyDataOrigin.Socioeconomic)

    def __init__(self, name: str, origin: DummyDataOrigin):
        super().__init__(name)
        self._origin = origin
    
    @property
    def name(self) -> str:
        return self._name_

    @property
    def origin(self) -> DummyDataOrigin:
        return self._origin
    
    def to_path(self, base_path: str) -> str:
        """Returns a file path for this data type based on the provided base path."""
        return os.path.join(base_path, f"dummy_{self.origin}_{self.name}.csv")


In [ ]:
class DummyConfig:
    def __init__(self, data_type: DummyDataType, num_points: int = DEFAULT_NUM_POINTS):
        self.data_type = data_type
        self.num_points = num_points
    
    def generate_dataset(self, output_dir: Optional[str] = None) -> pd.DataFrame:
        """Generates a dummy dataset based on the configuration. Also saves it to a CSV file if output_dir is provided."""
        coordinates = sample_land_coordinates(self.num_points)
        data = []
        for x, y in coordinates:
            data.append({
                "x": x,
                "y": y,
                "value": random.random(),  # random value between 0 and 1
                "data_type": self.data_type.value[0],
                "origin": self.data_type.value[1].value
            })
        df = pd.DataFrame(data)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            df.to_csv(self.data_type.to_path(output_dir), index=True, index_label="id")
        return df

## Custom Config

In [ ]:
config: Dict[DummyDataType, DummyConfig] = {
    DummyDataType.Fire: DummyConfig(DummyDataType.Fire, num_points=10_000),
}

In [ ]:
# create datasets based on config and default values and save them to CSV files
for data_type in DummyDataType:
    if data_type in config:
        cur_conf = config[data_type]
    else:
        cur_conf = DummyConfig(data_type)
    cur_conf.generate_dataset(output_dir=DUMMY_DATASET_BASE_PATH)

# Plot Dummy Data for Debugging

In [ ]:
def plot_dummy_data(dummy_src: Union[DummyConfig, DummyDataType], create_temp: bool = True):
    """
    Plots the dummy data for the given data type.
    If dummy_src is a DummyDataType, it will read the dataset from the corresponding CSV file.
    If dummy_src is a DummyConfig, it will generate a new dataset based on the configuration.
    """
    if isinstance(dummy_src, DummyDataType):
        data_type = dummy_src
        df = pd.read_csv(data_type.to_path(DUMMY_DATASET_BASE_PATH))
    else:
        data_type = dummy_src.data_type
        df = dummy_src.generate_dataset()

    plt.figure(figsize=(10, 6))
    plt.scatter(df["x"], -df["y"], c=df["value"], cmap="viridis", s=10)     # we need to use -y to flip the coordinates since the texture's zero point is top left, while matplotlib's is bottom left
    plt.colorbar(label="Value")
    plt.title(f"Dummy Data - {data_type.name} ({data_type.origin})")
    plt.xlabel("X Coordinate")
    plt.ylabel("Y Coordinate")
    plt.show()

In [ ]:
plot_dummy_data(DummyConfig(DummyDataType.Fire, num_points=1000))